In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense

In [2]:
# 1. Training data
pairs = [
    ("hello", "bonjour"),
    ("good morning", "bonjour"),
    ("thank you", "merci"),
    ("good night", "bonne nuit"),
    ("i am happy", "je suis heureux"),
    ("i am sad", "je suis triste"),
    ("how are you", "comment allez vous"),
    ("see you", "a bientot")
]

In [3]:
# 2. Create vocabulary
src_words = sorted(set(" ".join(x for x, y in pairs).split()))
tgt_words = sorted(set(" ".join(y for x, y in pairs).split()))

src_vocab = {w: i+1 for i, w in enumerate(src_words)}
tgt_vocab = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2}
tgt_vocab.update({w: i+3 for i, w in enumerate(tgt_words)})
reverse_vocab = {i: w for w, i in tgt_vocab.items()}

In [4]:
# 3. Convert sentences into sequences
encoder_data = [[src_vocab[w] for w in x.split()] for x, y in pairs]

decoder_input = [[1] + [tgt_vocab[w] for w in y.split()] for x, y in pairs]
decoder_output = [[tgt_vocab[w] for w in y.split()] + [2] for x, y in pairs]

max_src = max(map(len, encoder_data))
max_tgt = max(map(len, decoder_input))

encoder_data = pad_sequences(encoder_data, maxlen=max_src, padding="post")
decoder_input = pad_sequences(decoder_input, maxlen=max_tgt, padding="post")
decoder_output = pad_sequences(decoder_output, maxlen=max_tgt, padding="post")


In [5]:
# 4. Encoder
enc_input = Input(shape=(max_src,))
enc_embed = Embedding(len(src_vocab)+1, 32)(enc_input)
_, h, c = LSTM(64, return_state=True)(enc_embed)

# 5. Decoder
dec_input = Input(shape=(max_tgt,))
dec_embed = Embedding(len(tgt_vocab), 32)(dec_input)
dec_out = LSTM(64, return_sequences=True)(dec_embed, initial_state=[h, c])
output = Dense(len(tgt_vocab), activation="softmax")(dec_out)

In [6]:
# 6. Build and train model
model = Model([enc_input, dec_input], output)
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

model.fit([encoder_data, decoder_input],
          decoder_output[..., None],
          epochs=300, verbose=0)


In [7]:
# 7. Evaluate model
loss, accuracy = model.evaluate(
    [encoder_data, decoder_input],
    decoder_output[..., None], verbose=0
)
print("Accuracy:", round(accuracy * 100, 2), "%")

Accuracy: 100.0 %


In [8]:
# 8. Generate translated output
for i, (english, french) in enumerate(pairs[:4]):
    prediction = model.predict(
        [encoder_data[i:i+1], decoder_input[i:i+1]], verbose=0
    )
    ids = np.argmax(prediction[0], axis=1)
    words = [reverse_vocab[x] for x in ids if x > 2]
    print(english, "->", " ".join(words))


hello -> bonjour
good morning -> bonjour
thank you -> merci
good night -> bonne nuit
